# 06 — Kamera Live & Log Absensi (menulis `live_absensi_v2.py`)**Tujuan:** membuat aplikasi absensi berbasis kamera yang berjalan sebagai **script standalone di CMD** (bukan di dalam notebook), lalu memverifikasi bahwa script tersebut benar-benar bisa dijalankan.**Mengapa script, bukan `cv2.imshow` di notebook?** PRD §19.4: `cv2.imshow` tidak dipakai di dalam notebook. Notebook ini hanya **menulis** file `live_absensi_v2.py` secara lengkap (mengikuti pola SEL 18 di notebook lama), lalu memberi instruksi menjalankannya.**Prasyarat** — notebook 03 & 04 sudah dijalankan:- `Kapita\gallery.npz`, `Kapita\gallery_meta.json`- `Kapita\threshold.txt`**Isi aplikasi (memenuhi FR-1 s.d. FR-8):**| FR | Fitur di script ||---|---|| FR-1 | mengenali 69 orang dari kamera (MTCNN + VGGFace2 embedding) || FR-2 | wajah tak dikenal → `"Tidak dikenal"` || FR-3 | log CSV: timestamp, nama, NIM, kelas, similarity, status || FR-5 | jendela kamera menampilkan nama + skor; **tekan `Q` untuk keluar** || FR-6 | ambang dibaca dari `threshold.txt` (bisa diubah tanpa edit kode) || FR-7 | rekap per tanggal & per kelas (`--rekap`) || FR-8 | hanya centroid `aktif=True` yang dipakai |**Urutan eksekusi:** SEL 1 → SEL 4 berurutan, dari kernel kosong.**Menjalankan:**```batpy "./data_riset/raw\live_absensi_v2.py"REM rekap absensi hari ini:py "./data_riset/raw\live_absensi_v2.py" --rekapREM pilih foto dari laptop lewat dialog Windows (bisa pilih banyak):py "./data_riset/raw\live_absensi_v2.py" --pilih --banyakREM proses semua foto dalam satu folder:py "./data_riset/raw\live_absensi_v2.py" --folder "C:\path\foto" --tanpa-jendelaREM simulasi satu foto dari path tertentu:py "./data_riset/raw\live_absensi_v2.py" --uji "C:\path\foto.jpg"```

In [ ]:
# SEL 1 — Cek prasyarat: gallery + ambang harus sudah ada
import os
import sys
import json
from datetime import datetime

DATA_ROOT = r"./data_riset/raw"
GALLERY_NPZ = os.path.join(DATA_ROOT, "gallery.npz")
GALLERY_META = os.path.join(DATA_ROOT, "gallery_meta.json")
THRESHOLD_FILE = os.path.join(DATA_ROOT, "threshold.txt")
SCRIPT_OUT = os.path.join(DATA_ROOT, "live_absensi_v2.py")
LOG_DIR = DATA_ROOT
CAMERA_INDEX = 0
DEBOUNCE_DETIK = 60        # satu orang dicatat sekali per 60 detik (§20.C)
MAX_SIDE = 1600
CROP_FACTOR = 1.15         # WAJIB sama dengan notebook 01 (PRD §8.4 no. 13)
ROLL_CORRECT = True
IMG_SIZE = 160
SEED = 42


def baca_threshold(path=THRESHOLD_FILE):
    nilai, meta = None, {}
    with open(path, encoding="utf-8") as fh:
        for baris in fh:
            baris = baris.strip()
            if not baris or baris.startswith("#"):
                continue
            if "=" in baris:
                k, v = baris.split("=", 1)
                meta[k.strip()] = v.strip()
            else:
                nilai = float(baris)
    return nilai, meta


for p in (GALLERY_NPZ, GALLERY_META, THRESHOLD_FILE):
    print("{:<52} {}".format(os.path.basename(p), "ada" if os.path.isfile(p) else "HILANG -> jalankan notebook sebelumnya"))
assert os.path.isfile(GALLERY_NPZ), "gallery.npz belum ada"
assert os.path.isfile(GALLERY_META), "gallery_meta.json belum ada"
assert os.path.isfile(THRESHOLD_FILE), "threshold.txt belum ada"

nilai, meta_th = baca_threshold()
with open(GALLERY_META, encoding="utf-8") as fh:
    meta = json.load(fh)
print("ambang             :", nilai)
print("orang terdaftar    :", meta["jumlah_orang"], "| aktif:", meta["jumlah_aktif"])
print("script akan ditulis:", SCRIPT_OUT)

In [ ]:
# SEL 2 — Tulis file live_absensi_v2.py (script lengkap, siap jalan)
KODE_SCRIPT = r'''
# -*- coding: utf-8 -*-
# live_absensi_v2.py - Sistem absensi berbasis pengenalan wajah (Fase 2)
# Dibuat oleh notebook 06_live_absensi.ipynb. Jalankan dari CMD:
# py "./data_riset/raw\live_absensi_v2.py
# py "./data_riset/raw\live_absensi_v2.py" --rekap
# py "./data_riset/raw\live_absensi_v2.py" --uji "C:\path\foto.jpg
# py "./data_riset/raw\live_absensi_v2.py" --kamera 1
# Tekan Q pada jendela kamera untuk keluar.
# Preprocessing WAJIB identik dengan notebook 01 (PRD §11 & §8.4 no. 13).

import argparse
import csv
import json
import os
import sys
import time
from collections import defaultdict
from datetime import datetime

import numpy as np
import cv2
import torch
import torch.nn.functional as F
import pillow_heif
from PIL import Image
from facenet_pytorch import MTCNN, InceptionResnetV1

try:
    import tkinter as tk
    from tkinter import filedialog
    ADA_DIALOG = True
except Exception:
    ADA_DIALOG = False

# pillow-heif: file .jpg yang sebenarnya HEIC (kamera iPhone) tetap terbaca
pillow_heif.register_heif_opener()

# ------------------------- KONSTANTA (bisa diubah tanpa edit logika) -------
DATA_ROOT = r"./data_riset/raw"
GALLERY_NPZ = os.path.join(DATA_ROOT, "gallery.npz")
GALLERY_META = os.path.join(DATA_ROOT, "gallery_meta.json")
THRESHOLD_FILE = os.path.join(DATA_ROOT, "threshold.txt")
LOG_DIR = DATA_ROOT
CAMERA_INDEX = 0
DEBOUNCE_DETIK = 60
MAX_SIDE = 1600
CROP_FACTOR = 1.15
ROLL_CORRECT = True
IMG_SIZE = 160
LABEL_TIDAK_DIKENAL = "Tidak dikenal"


# ------------------------------- UTILITAS -----------------------------------
def baca_threshold(path=THRESHOLD_FILE):
    # Ambang dari file: baris pertama = nilai, baris 'key=value' = metadata.

    nilai, meta = None, {}
    with open(path, encoding="utf-8") as fh:
        for baris in fh:
            baris = baris.strip()
            if not baris or baris.startswith("#"):
                continue
            if "=" in baris:
                k, v = baris.split("=", 1)
                meta[k.strip()] = v.strip()
            else:
                nilai = float(baris)
    if nilai is None:
        nilai = 0.68
    return nilai, meta


def imread_unicode(path):
    # Baca gambar dari path Windows mana pun (aman non-ASCII + fallback HEIC).

    img = None
    try:
        img = cv2.imdecode(np.fromfile(path, dtype=np.uint8), cv2.IMREAD_COLOR)
    except Exception:
        img = None
    if img is None:
        try:
            with Image.open(path) as pil:
                img = cv2.cvtColor(np.array(pil.convert("RGB")), cv2.COLOR_RGB2BGR)
        except Exception:
            return None
    return img


def log_path(tanggal=None):
    d = tanggal or datetime.now().strftime("%Y-%m-%d")
    return os.path.join(LOG_DIR, "absensi_" + d + ".csv")


KOLOM_LOG = ["timestamp", "nama", "nim", "kelas", "similarity", "status"]


def catat_absensi(similarity, status, nama, nim, kelas):
    # Tulis satu baris ke log CSV hari ini (format PRD §20.C).

    path = log_path()
    baru = not os.path.isfile(path)
    with open(path, "a", newline="", encoding="utf-8-sig") as fh:
        writer = csv.writer(fh)
        if baru:
            writer.writerow(KOLOM_LOG)
        writer.writerow([datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
                         nama, nim or "-", kelas or "-",
                         "{:.4f}".format(float(similarity)), status])
    return path


def sudah_dicatat(path, nama, window_detik=DEBOUNCE_DETIK):
    # True bila nama sudah tercatat di dalam window debounce (FR-3 & PRD 20.C)
    if not os.path.isfile(path):
        return False
    sekarang = datetime.now()
    with open(path, "r", encoding="utf-8-sig") as fh:
        for row in csv.DictReader(fh):
            if row.get("nama") != nama:
                continue
            try:
                ts = datetime.strptime(row["timestamp"], "%Y-%m-%d %H:%M:%S")
            except Exception:
                continue
            if abs((sekarang - ts).total_seconds()) < window_detik:
                return True
    return False


# ------------------------- PREPROCESSING (KANONIK) -------------------------
def downscale_max_side(img, max_side=MAX_SIDE):
    h, w = img.shape[:2]
    if max(h, w) > max_side:
        s = max_side / float(max(h, w))
        img = cv2.resize(img, (int(round(w * s)), int(round(h * s))), interpolation=cv2.INTER_AREA)
    return img


def roll_degrees(landmarks5):
    pts = np.asarray(landmarks5, dtype=np.float32).reshape(-1, 2)
    d = pts[1] - pts[0]
    return float(np.degrees(np.arctan2(float(d[1]), float(d[0]))))


def align_face(bgr, box, landmarks5, canvas=IMG_SIZE, crop_factor=CROP_FACTOR,
               roll_correct=ROLL_CORRECT):
    # Crop bujur sangkar dari bbox DETEKTOR + koreksi roll.

    x0, y0, x1, y1 = [float(v) for v in box]
    cx, cy = (x0 + x1) / 2.0, (y0 + y1) / 2.0
    side = max(x1 - x0, y1 - y0) * float(crop_factor)
    img = bgr
    if roll_correct:
        sudut = roll_degrees(landmarks5)
        if abs(sudut) > 0.5:
            M = cv2.getRotationMatrix2D((cx, cy), sudut, 1.0)
            img = cv2.warpAffine(img, M, (img.shape[1], img.shape[0]),
                                 flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_REPLICATE)
    h, w = img.shape[:2]
    sx1 = int(max(0, round(cx - side / 2)))
    sy1 = int(max(0, round(cy - side / 2)))
    sx2 = int(min(w, round(cx + side / 2)))
    sy2 = int(min(h, round(cy + side / 2)))
    if sx2 - sx1 < 8 or sy2 - sy1 < 8:
        return None
    return cv2.resize(img[sy1:sy2, sx1:sx2], (canvas, canvas), interpolation=cv2.INTER_AREA)


def preprocess_embedding(bgr):
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB).astype(np.float32)
    return torch.from_numpy((rgb - 127.5) / 128.0).permute(2, 0, 1)


# ------------------------------- MODEL -------------------------------------
class Pengenal:
    def __init__(self, device=None):
        self.device = device or torch.device("cuda" if torch.cuda.is_available() else "cpu")
        g = np.load(GALLERY_NPZ, allow_pickle=False)
        aktif = g["aktif"].astype(bool)
        self.centroid = g["embeddings"].astype(np.float32)[aktif]
        self.person = g["person"].astype(str)[aktif]
        with open(GALLERY_META, encoding="utf-8") as fh:
            meta = json.load(fh)
        info = {o["person"]: o for o in meta["orang"]}
        self.nama = [info[p]["nama"] if p in info else p for p in self.person]
        self.nim = [info[p]["nim"] if p in info else "" for p in self.person]
        self.kelas = [info[p]["kelas"] if p in info else "-" for p in self.person]
        # Label tampilan di kamera: NAMA_NIM_KELAS (ter-normalisasi, PRD 20.C)
        # Nama folder bisa terbalik urutannya (mis. "20260001_Nama"),
        # jadi label dibangun dari metadata, bukan dari nama folder.
        self.label = []
        for i, pid in enumerate(self.person):
            nim = self.nim[i] if self.nim[i] else "-"
            kls = self.kelas[i] if self.kelas[i] else "-"
            self.label.append("{}_{}_{}".format(self.nama[i], nim, kls))
        self.mtcnn = MTCNN(keep_all=True, select_largest=True, min_face_size=20,
                           device=self.device)
        self.net = InceptionResnetV1(pretrained=str(g["model"][0])).eval().to(self.device)
        for prm in self.net.parameters():
            prm.requires_grad_(False)
        self.threshold, self.meta_th = baca_threshold()
        print("[init] device            :", self.device)
        print("[init] orang terdaftar   :", len(self.person))
        print("[init] ambang similarity :", self.threshold)
        print("[init] log hari ini      :", log_path())

    @torch.no_grad()
    def embed(self, faces):
        x = torch.stack([preprocess_embedding(f) for f in faces]).to(self.device)
        return F.normalize(self.net(x), dim=1).cpu().numpy().astype(np.float32)

    @torch.no_grad()
    def kenali(self, bgr):
        # Frame/photo -> list dict {nama, similarity, box, dikenali}.

        img = downscale_max_side(bgr)
        rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        boxes, probs, points = self.mtcnn.detect(rgb, landmarks=True)
        hasil = []
        if boxes is None or len(boxes) == 0 or points is None:
            return hasil, img
        boxes = np.asarray(boxes, dtype=np.float32)
        points = np.asarray(points, dtype=np.float32)
        area = (boxes[:, 2] - boxes[:, 0]) * (boxes[:, 3] - boxes[:, 1])
        for i in np.argsort(-area):
            face = align_face(img, boxes[i], points[i])
            if face is None:
                continue
            emb = self.embed([face])[0]
            sim = self.centroid @ emb
            j = int(np.argmax(sim))
            sk = float(sim[j])
            kena = sk >= self.threshold
            hasil.append({
                "label": self.label[j] if kena else LABEL_TIDAK_DIKENAL,
                "nama": self.nama[j] if kena else LABEL_TIDAK_DIKENAL,
                "person": self.person[j] if kena else "",
                "nim": self.nim[j] if kena else "",
                "kelas": self.kelas[j] if kena else "",
                "similarity": sk,
                "dikenali": bool(kena),
                "box": boxes[i].tolist(),
            })
        return hasil, img


# --------------------------------- REKAP -----------------------------------
def rekap(tanggal=None):
    # FR-7: rekap absensi per tanggal & per kelas.

    path = log_path(tanggal)
    if not os.path.isfile(path):
        print("Belum ada log untuk", tanggal or datetime.now().strftime("%Y-%m-%d"))
        return
    rows = []
    with open(path, "r", encoding="utf-8-sig") as fh:
        for row in csv.DictReader(fh):
            rows.append(row)
    hadir = [r for r in rows if r["status"] == "HADIR"]
    ditolak = [r for r in rows if r["status"] != "HADIR"]
    per_kelas = defaultdict(list)
    for r in hadir:
        per_kelas[r["kelas"]].append(r["nama"])
    print("=" * 60)
    print("REKAP ABSENSI -", tanggal or datetime.now().strftime("%Y-%m-%d"))
    print("=" * 60)
    print("total catatan :", len(rows))
    print("hadir         :", len(hadir), "orang unik:", len(set(r["nama"] for r in hadir)))
    print("ditolak       :", len(ditolak))
    print("\nper kelas:")
    for k in sorted(per_kelas):
        print("  {:<14} {:>3} orang".format(k, len(set(per_kelas[k]))))
    print("\ndaftar hadir:")
    for r in sorted(hadir, key=lambda x: x["timestamp"]):
        print("  {} | {:<32} | {:<16} | sim {}".format(
            r["timestamp"], r["nama"][:32], r["kelas"], r["similarity"]))


# ---------------------- PILIH FILE DARI LAPTOP (FR-9) ----------------------
JENIS_GAMBAR = [
    ("Foto (*.jpg *.jpeg *.png *.bmp *.webp)", "*.jpg *.jpeg *.png *.bmp *.webp"),
    ("Semua file (*.*)", "*.*"),
]


def pilih_file(jumlah_satu=True):
    # Buka dialog pemilih file Windows. Return list path (boleh kosong).
    if ADA_DIALOG:
        try:
            root = tk.Tk()
            root.withdraw()
            root.attributes("-topmost", True)
            if jumlah_satu:
                path = filedialog.askopenfilename(
                    title="Pilih foto untuk dideteksi absensi",
                    filetypes=JENIS_GAMBAR)
                daftar = [path] if path else []
            else:
                daftar = list(filedialog.askopenfilenames(
                    title="Pilih satu atau beberapa foto (bisa pilih banyak)",
                    filetypes=JENIS_GAMBAR))
            root.destroy()
            return daftar
        except Exception as e:
            print("[pilih] dialog gagal:", e)
    # cadangan: ketik path manual
    try:
        masuk = input("Masukkan path foto (Enter untuk batal): ").strip().strip(chr(34))
    except Exception:
        return []
    return [masuk] if masuk else []


def gambar_dalam_folder(folder):
    # Semua file gambar dalam folder (1 level).
    if not os.path.isdir(folder):
        return []
    ext = (".jpg", ".jpeg", ".png", ".bmp", ".webp")
    return [os.path.join(folder, f) for f in sorted(os.listdir(folder))
            if f.lower().endswith(ext)]


def gambar_dengan_label(pengenal, path, tampilkan=True):
    # Proses 1 foto -> (gambar_berannotasi, hasil).
    img = imread_unicode(path)
    if img is None:
        print("Gagal membaca:", path)
        return None, None
    hasil, img = pengenal.kenali(img)
    vis = img.copy()
    for h in hasil:
        x0, y0, x1, y1 = [int(v) for v in h["box"]]
        warna = (0, 220, 0) if h["dikenali"] else (0, 0, 255)
        cv2.rectangle(vis, (x0, y0), (x1, y1), warna, 2)
        label = "{} | {:.3f}".format(h["label"][:58], h["similarity"])
        (tw, th_), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 2)
        tx = max(2, min(x0, vis.shape[1] - tw - 6))
        ty = max(18, y0 - 8)
        cv2.rectangle(vis, (tx - 3, ty - th_ - 6), (tx + tw + 3, ty + 6), (0, 0, 0), -1)
        cv2.putText(vis, label, (tx, ty), cv2.FONT_HERSHEY_SIMPLEX, 0.5, warna, 1, cv2.LINE_AA)
    if not hasil:
        print("  tidak ada wajah terdeteksi")
    else:
        for h in hasil:
            print("  {} | similarity {:.4f} | {}".format(
                h["label"], h["similarity"], "DIKENAL" if h["dikenali"] else "DITOLAK"))
            catat_absensi(h["similarity"], "HADIR" if h["dikenali"] else "DITOLAK",
                          h["nama"], h["nim"], h["kelas"])
    if tampilkan and hasil:
        try:
            cv2.imshow(os.path.basename(path)[:60], vis)
            cv2.waitKey(0)
            cv2.destroyAllWindows()
        except Exception as e:
            print("  (tidak bisa menampilkan jendela:", e, ")")
    return vis, hasil


def proses_semua(pengenal, daftar, tampilkan=True):
    # Proses daftar file, tampilkan ringkasan di akhir.
    if not daftar:
        print("Tidak ada file dipilih.")
        return
    print("Memproses {} file...".format(len(daftar)))
    rekap = []
    for n, path in enumerate(daftar, start=1):
        print("\n[{}/{}] {}".format(n, len(daftar), os.path.basename(path)))
        vis, hasil = gambar_dengan_label(pengenal, path, tampilkan=tampilkan)
        if hasil:
            terbaik = hasil[0]
            rekap.append((os.path.basename(path), terbaik["label"],
                          terbaik["similarity"], terbaik["dikenali"]))
    print("\n" + "=" * 70)
    print("RINGKASAN")
    print("=" * 70)
    print("{:<34} {:<52} {:>7}".format("file", "hasil", "similarity"))
    for f, lab, sim, kena in rekap:
        print("{:<34} {:<52} {:>7.3f}  {}".format(f[:33], lab[:51], sim,
                                                   "HADIR" if kena else "DITOLAK"))
    print("\nLog absensi:", log_path())


# ------------------------------- APLIKASI ----------------------------------
def jalankan_kamera(pengenal, camera_index=CAMERA_INDEX):
    cap = cv2.VideoCapture(camera_index)
    if not cap.isOpened():
        print("Kamera tidak bisa dibuka. Ubah --kamera atau tutup aplikasi kamera lain.")
        return
    lebar = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)) or 640
    tinggi = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)) or 480
    print("\n[kamera] mulai berjalan. Tekan Q pada jendela untuk keluar.")
    t_terakhir = 0.0
    path_log = log_path()
    try:
        while True:
            ok, frame = cap.read()
            if not ok:
                break
            frame = cv2.resize(frame, (lebar, tinggi))
            hasil, img = pengenal.kenali(frame)
            for h in hasil:
                x0, y0, x1, y1 = [int(v) for v in h["box"]]
                warna = (0, 220, 0) if h["dikenali"] else (0, 0, 255)
                cv2.rectangle(img, (x0, y0), (x1, y1), warna, 2)
                # tampilkan NAMA_NIM_KELAS di atas kotak (PRD 20.C)
                label = "{} | {:.3f}".format(h["label"][:58], h["similarity"])
                (tw, th_), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 2)
                tx = max(2, min(x0, lebar - tw - 6))
                ty = max(18, y0 - 8)
                cv2.rectangle(img, (tx - 3, ty - th_ - 6), (tx + tw + 3, ty + 6), (0, 0, 0), -1)
                cv2.putText(img, label, (tx, ty), cv2.FONT_HERSHEY_SIMPLEX,
                            0.5, warna, 1, cv2.LINE_AA)
                sekarang = time.time()
                if sekarang - t_terakhir > 0.8 and not sudah_dicatat(path_log, h["nama"]):
                    catat_absensi(h["similarity"],
                                  "HADIR" if h["dikenali"] else "DITOLAK",
                                  h["nama"], h["nim"], h["kelas"])
                    t_terakhir = sekarang
            cv2.putText(img, "Q = keluar", (10, tinggi - 12), cv2.FONT_HERSHEY_SIMPLEX,
                        0.5, (255, 255, 255), 1, cv2.LINE_AA)
            cv2.imshow("Absensi Wajah - tekan Q untuk keluar", img)
            if cv2.waitKey(1) & 0xFF in (ord("q"), ord("Q")):
                print("\n[kamera] berhenti. Terima kasih!")
                break
    finally:
        cap.release()
        cv2.destroyAllWindows()


def uji_foto(pengenal, path):
    # Simulasi kamera dari satu foto (tanpa webcam) - berguna untuk demo A4.

    img = imread_unicode(path)
    if img is None:
        print("Gagal membaca:", path)
        return
    hasil, _ = pengenal.kenali(img)
    if not hasil:
        print("Tidak ada wajah terdeteksi.")
        return
    for h in hasil:
        print("{} | similarity {:.4f} | {}".format(
            h["label"], h["similarity"], "DIKENAL" if h["dikenali"] else "DITOLAK"))
        catat_absensi(h["similarity"], "HADIR" if h["dikenali"] else "DITOLAK",
                      h["nama"], h["nim"], h["kelas"])
    print("Log:", log_path())


def main():
    ap = argparse.ArgumentParser(description="Absensi berbasis pengenalan wajah")
    ap.add_argument("--kamera", type=int, default=CAMERA_INDEX, help="index kamera (default 0)")
    ap.add_argument("--rekap", action="store_true", help="tampilkan rekap hari ini lalu keluar")
    ap.add_argument("--tanggal", default=None, help="tanggal rekap, format YYYY-MM-DD")
    ap.add_argument("--uji", default=None, help="path foto untuk disimulasikan (tanpa webcam)")
    ap.add_argument("--pilih", action="store_true",
                    help="buka dialog pilih file foto dari laptop, lalu deteksi")
    ap.add_argument("--banyak", action="store_true",
                    help="boleh pilih beberapa file sekaligus (dipakai bersama --pilih)")
    ap.add_argument("--folder", default=None,
                    help="proses semua foto dalam satu folder")
    ap.add_argument("--tanpa-jendela", action="store_true",
                    help="tidak menampilkan jendela hasil (untuk batch)")
    args = ap.parse_args()

    if args.rekap:
        rekap(args.tanggal)
        return
    pengenal = Pengenal()
    tampilkan = not args.tanpa_jendela

    if args.pilih:
        daftar = pilih_file(jumlah_satu=not args.banyak)
        if not daftar:
            print("Tidak ada file dipilih - keluar.")
            return
        proses_semua(pengenal, daftar, tampilkan=tampilkan)
    elif args.folder:
        daftar = gambar_dalam_folder(args.folder)
        if not daftar:
            print("Folder tidak berisi foto:", args.folder)
            return
        proses_semua(pengenal, daftar, tampilkan=tampilkan)
    elif args.uji:
        uji_foto(pengenal, args.uji)
    else:
        jalankan_kamera(pengenal, args.kamera)


if __name__ == "__main__":
    main()
'''

with open(SCRIPT_OUT, "w", encoding="utf-8") as fh:
    fh.write(KODE_SCRIPT)

import py_compile
py_compile.compile(SCRIPT_OUT, doraise=True)
print("script ditulis :", SCRIPT_OUT, "|", os.path.getsize(SCRIPT_OUT), "byte")
print("sintaks valid  : OK (py_compile)")

In [ ]:
# SEL 3 — Verifikasi isi script:artefak wajib + smoke test logika
import subprocess
import sys as _sys

isi = open(SCRIPT_OUT, encoding="utf-8").read()
wajib = [
    ("baca gallery.npz", "np.load(GALLERY_NPZ"),
    ("baca gallery_meta.json", "GALLERY_META"),
    ("baca threshold.txt", "def baca_threshold"),
    ("MTCNN", "MTCNN(keep_all=True"),
    ("InceptionResnetV1", "InceptionResnetV1"),
    ("normalisasi (x-127.5)/128", "(rgb - 127.5) / 128.0"),
    ("crop factor 1.15", "CROP_FACTOR = 1.15"),
    ("koreksi roll", "def roll_degrees"),
    ("cosine similarity", "self.centroid @ emb"),
    ("label NAMA_NIM_KELAS", "self.label"),
    ("label difit ke lebar", "cv2.getTextSize"),
    ("label Tidak dikenal", 'LABEL_TIDAK_DIKENAL = "Tidak dikenal"'),
    ("log CSV", "def catat_absensi"),
    ("debounce 60 detik", "DEBOUNCE_DETIK = 60"),
    ("tekan Q keluar", 'ord("q")'),
    ("rekap per kelas", "def rekap"),
    ("hanya yang aktif", "aktif = g[\"aktif\"]"),
    ("argparse --uji", "add_argument(\"--uji\""),
    ("dialog pilih file", "def pilih_file"),
    ("mode --pilih", "add_argument(\"--pilih\""),
    ("mode --folder", "add_argument(\"--folder\""),
    ("mode --banyak", "add_argument(\"--banyak\""),
    ("mode --tanpa-jendela", "add_argument(\"--tanpa-jendela\""),
]
print("pemeriksaan isi script:")
ok_semua = True
for nama,kunci in wajib:
    ada =kunci in isi
    ok_semua = ok_semua and ada
    print("  {:<32} {}".format(nama, "OK" if ada else "HILANG"))
assert ok_semua, "ada bagian penting yang hilang dari live_absensi_v2.py"

print("\nmenjalankan --rekap (tidak butuh kamera):")
hasil = subprocess.run([_sys.executable, SCRIPT_OUT, "--rekap"], capture_output=True, text=True)
print(hasil.stdout.strip() or "(belum ada log absensi hari ini)")
if hasil.returncode != 0:
    print("STDERR:", hasil.stderr.strip()[:800])
assert hasil.returncode == 0, "script gagal dijalankan dengan --rekap"
print("\nOK: script bisa dijalankan tanpa kamera (mode --rekap).")

In [ ]:
# SEL 4 — Uji aplikasi pada foto nyata (A4 tanpa webcam)
import glob
import random

# cari foto bersih (varian 'orig') milik SEJAK beberapa orang untuk simulasi
kandidat = sorted(glob.glob(os.path.join(r"./data_riset/aug\augmented", "*", "*_orig.jpg")))
print("foto tersimpan untuk simulasi:", len(kandidat))
if not kandidat:
    print("Belum ada hasil notebook 02 -> jalankan dulu.")
else:
    random.seed(SEED)
    contoh = random.sample(kandidat, min(5, len(kandidat)))
    import subprocess
    import sys as _sys
    for p in contoh:
        print("\n==>", os.path.basename(os.path.dirname(p)), "|", os.path.basename(p))
        r = subprocess.run([_sys.executable, SCRIPT_OUT, "--uji", p], capture_output=True, text=True)
        print(r.stdout.strip() or r.stderr.strip()[:400])
    print("\nLog absensi sekarang:", os.path.join(DATA_ROOT, "absensi_2026-10-06.csv"), "atau file hari ini")

# Langkah menjalankan demo (A4 & A5)1. Pastikan `gallery.npz`, `gallery_meta.json`, `threshold.txt` ada (sudah dibuat notebook 03 & 04).2. Jalankan kamera:```batpy "./data_riset/raw\live_absensi_v2.py"```3. Jendela akan menampilkan kotak hijau (dikenali) atau merah (tidak dikenal) + nama + skor similarity.4. Setiap orang yang dikenali **otomatis tercatat** ke `absensi_YYYY-MM-DD.csv` — maksimal sekali per 60 detik.5. Tekan **Q** untuk keluar.6. Lihat rekap:```batpy "./data_riset/raw\live_absensi_v2.py" --rekap```**Kriteria yang bisa diverifikasi lewat script ini:**- **A4** — wajah pengguna uji dikenali sebagai dirinya dengan similarity ≥ 0,68.- **A5** — file `absensi_YYYY-MM-DD.csv` terisi otomatis dengan kolom PRD §20.C.**Kalau similarity sering di bawah ambang:**- Naikkan `MARGIN` cahaya, jangan terlalu jauh dari kamera (60–80 cm).- Periksa dengan `--uji` pada foto yang gagal dikenali.- Opsional: turunkan ambang di `threshold.txt` (baris pertama) — tapi ingat FAR naik (lihat tabel trade-off di notebook 04).